# Ahmed Mohamed Fahmy 231000587
# Myar Saad Sadek 231000687
# Nadeem Diaa Shokry 231000857
# Omar Salama Isleem 231000674

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
files = [
    "CDR.csv",
    "PTDEMOG.csv",
    "APOERES.csv",
    "MMSE.csv",
    "DXSUM.csv",
    "FAQ.csv",
    "FOXLABBSI.csv",
    "ADAS.csv",
    "NEUROBAT.csv",
    "UCD_WMH.csv"
]
raw_path = "/content/drive/MyDrive/Alzheimer-Progression-Prediction/Data/Raw/"
processed_path = "/content/drive/MyDrive/Alzheimer-Progression-Prediction/Data/Processed/"
dictionary = pd.read_csv( raw_path + 'DATADIC.csv')
dfs = {}

for csv_file in files:
  df_name = csv_file.replace('.csv', '')
  dfs[df_name] = pd.read_csv(raw_path + csv_file, low_memory=False)
  print(f"--- {df_name} --- ")
  print(f"Columns: {dfs[df_name].columns.tolist()}\n")
  print(f"Shape: {dfs[df_name].shape}\n")

--- CDR --- 
Columns: ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'CDSOURCE', 'CDVERSION', 'SPID', 'CDMEMORY', 'CDORIENT', 'CDJUDGE', 'CDCOMMUN', 'CDHOME', 'CDCARE', 'CDGLOBAL', 'CDRSB', 'ID', 'SITEID', 'USERDATE', 'USERDATE2', 'DD_CRF_VERSION_LABEL', 'LANGUAGE_CODE', 'HAS_QC_ERROR', 'update_stamp']

Shape: (14758, 25)

--- PTDEMOG --- 
Columns: ['PHASE', 'PTID', 'RID', 'VISCODE', 'VISCODE2', 'VISDATE', 'PTSOURCE', 'PTGENDER', 'PTDOB', 'PTDOBYY', 'PTHAND', 'PTMARRY', 'PTEDUCAT', 'PTWORKHS', 'PTWORK', 'PTNOTRT', 'PTRTYR', 'PTHOME', 'PTTLANG', 'PTPLANG', 'PTADBEG', 'PTCOGBEG', 'PTADDX', 'PTETHCAT', 'PTRACCAT', 'PTIDENT', 'PTORIENT', 'PTORIENTOT', 'PTENGSPK', 'PTNLANG', 'PTENGSPKAGE', 'PTCLANG', 'PTLANGSP', 'PTLANGWR', 'PTSPTIM', 'PTSPOTTIM', 'PTLANGPR1', 'PTLANGSP1', 'PTLANGRD1', 'PTLANGWR1', 'PTLANGUN1', 'PTLANGPR2', 'PTLANGSP2', 'PTLANGRD2', 'PTLANGWR2', 'PTLANGUN2', 'PTLANGPR3', 'PTLANGSP3', 'PTLANGRD3', 'PTLANGWR3', 'PTLANGUN3', 'PTLANGPR4', 'PTLANGSP4', 'PTLANGRD4', '

In [ ]:
final_columns_to_keep = {

    "CDR": [
        "RID",
        "VISCODE",
        "CDRSB"
    ],
    "PTDEMOG": [
        "RID",
        "PTGENDER",
        "PTEDUCAT",
        "PTDOB",
        "PTMARRY"],
    "APOERES":[
        "RID",
        "GENOTYPE"
    ],
    "MMSE":[
        "RID",
        "VISCODE",
        "MMSCORE" # Corrected from MMSE to MMSCORE
    ],
    "DXSUM": [
        "RID",
        "VISCODE",
        "EXAMDATE",
        "DIAGNOSIS", #1=CN;2=MCI;3=Dementia
        "DXAD",# 1=Yes AD
        "DXMCI",# 1=Yes MCI
    ],
    "NEUROBAT": [
        "RID",
        "VISCODE",
        # Memory (Core AD predictors)
        "LIMMTOTAL",          # Logical Memory Immediate
        "LDELTOTAL",          # Logical Memory Delayed
        "AVTOT1", "AVTOT2", "AVTOT3", "AVTOT4", "AVTOT5", "AVTOT6", # RAVLT Learning
        "AVDEL30MIN",         # RAVLT Delayed Recall (30 min)
        "AVDELTOT",           # RAVLT Recognition Score
        # Language / Semantic
        "CATANIMSC",          # Category Fluency Animals
    ],
    "ADAS":[
        "RID",
        "VISCODE",
        "TOTSCORE",
        "TOTAL13"
    ],
    "FAQ":[
        "RID",
        "FAQTOTAL"
    ],
    "FOXLABBSI": [
        "RID",
        "VISCODE",
        "BRAINVOL",
        "VENTVOL",
        ],
    "UCD_WMH": [
        "RID",
        "VISCODE",
        "TOTAL_HIPPO"
    ]
    }
dfs_filtered = {}
for table_name, cols in final_columns_to_keep.items():
    if table_name in dfs:
        # Ensure RID and VISCODE are included even if not explicitly listed (some tables use PTID)
        available_cols = [c for c in cols if c in dfs[table_name].columns]
        dfs_filtered[table_name] = dfs[table_name][available_cols].copy()
        print(f"{table_name}: kept {len(available_cols)} columns")
    else:
        print(f"Warning: {table_name} not found in loaded data")

CDR: kept 3 columns
PTDEMOG: kept 5 columns
APOERES: kept 2 columns
MMSE: kept 3 columns
DXSUM: kept 6 columns
NEUROBAT: kept 13 columns
ADAS: kept 4 columns
FAQ: kept 2 columns
FOXLABBSI: kept 4 columns
UCD_WMH: kept 3 columns


In [ ]:
df_column_descriptions = {}

for df_name, df in dfs_filtered.items():
    column_descriptions = {}
    # Convert df_name to uppercase to match TBLNAME in dictionary
    df_name_upper = df_name.upper()

    for column in df.columns:
        # Find the description in the dictionary
        description_row = dictionary[(dictionary['TBLNAME'] == df_name_upper) & (dictionary['FLDNAME'] == column)]

        if not description_row.empty:
            description = description_row['TEXT'].iloc[0]
            column_descriptions[column] = description
        else:
            column_descriptions[column] = 'No description found'

    df_column_descriptions[df_name] = column_descriptions

# Display the generated dictionaries for each dataframe
for df_name, descriptions in df_column_descriptions.items():
    print(f"--- Column Descriptions for {df_name} ---")
    for col, desc in descriptions.items():
        print(f"  {col}: {desc}")
    print("\n")

--- Column Descriptions for CDR ---
  RID: Participant roster ID
  VISCODE: Visit code
  CDRSB: CDR-SB


--- Column Descriptions for PTDEMOG ---
  RID: Participant roster ID
  PTGENDER: 1. Participant Gender
  PTEDUCAT: 5. Participant Education
  PTDOB: 2. Participant Date of Birth
  PTMARRY: 4. Participant Marital Status


--- Column Descriptions for APOERES ---
  RID: Participant roster ID
  GENOTYPE: Apolipoprotein E (Apo-E) Genotype


--- Column Descriptions for MMSE ---
  RID: Participant roster ID
  VISCODE: Visit code
  MMSCORE: MMSE TOTAL SCORE


--- Column Descriptions for DXSUM ---
  RID: Participant roster ID
  VISCODE: Visit code
  EXAMDATE: Date Form Completed
  DIAGNOSIS: 1.  Which best describes the participant's current diagnosis?
  DXAD: Alzheimer's Disease
  DXMCI: Mild Cognitive Impairment


--- Column Descriptions for NEUROBAT ---
  RID: Participant roster ID
  VISCODE: Visit code
  LIMMTOTAL: Logical Memory - Immediate Recall <br />Total Number of Story Units Recal

In [ ]:
def viscode_to_months(viscode, rid, phase):
    if pd.isna(viscode):
        return None

    viscode = str(viscode).strip().lower()
    phase   = str(phase).strip().upper() if pd.notna(phase) else ''

    # ── Screen fail ──────────────────────────────────────────────────────────
    if viscode == 'f':
        return -1

    # ── ADNI4 phase-prefixed codes ────────────────────────────────────────────
    adni4_map = {
        '4_sc':  0, '4_bl': 0, '4_init': 0,
        '4_m12': 12,
        '4_m24': 24,
    }
    if viscode in adni4_map:
        return adni4_map[viscode]

    # ── ADNI3 year codes ──────────────────────────────────────────────────────
    if phase == 'ADNI3':
        adni3_map = {
            'sc': 0, 'bl': 0, 'init': 0,
            'y1': 12, 'y2': 24, 'y3': 36, 'y4': 48, 'y5': 60,
        }
        if viscode in adni3_map:
            return adni3_map[viscode]

    # ── Universal baseline codes (ADNI1 / ADNIGO / ADNI2-new) ────────────────
    if viscode in ('sc', 'bl', 'scmri', 'init'):
        return 0

    # ── Direct month-coded visits: m06, m12, m18, m24 … (ADNI1/GO) ──────────
    if viscode.startswith('m'):
        try:
            return int(viscode[1:])
        except ValueError:
            return None

    # ── ADNI2 NEW participants (RID >= 4000) — static table from official doc ─
    # Source: ADNI2 VISCODE2 Assignment doc, Table 3
    if rid >= 4000 and phase == 'ADNI2':
        adni2_new_map = {
            'v01': 0,   # Screening
            'v02': 0,   # Screening MRI
            'v03': 0,   # Baseline
            'v04': 3,   # Month 3 MRI
            'v05': 6,   # Month 6
            'v11': 12,  # Year 1
            'v12': 18,  # Year 1 TelCheck
            'v21': 24,  # Year 2
            'v22': 30,  # Year 2 TelCheck
            'v31': 36,  # Year 3
            'v32': 42,  # Year 3 TelCheck
            'v41': 48,  # Year 4
            'v42': 54,  # Year 4 TelCheck
            'v51': 60,  # Year 5
            'v52': 66,  # Year 5 TelCheck
        }
        return adni2_new_map.get(viscode, None)


    if phase == 'ADNI2' and rid < 4000:
        return None

    # ── tau / uns1 and any other unrecognised codes ───────────────────────────
    return None


# Apply (requires PHASE column — present in NEUROBAT and most ADNI tables)
for name in dfs_filtered:
    if 'VISCODE' in dfs_filtered[name].columns:
        dfs_filtered[name]['MONTH'] = dfs_filtered[name].apply(
            lambda row: viscode_to_months(row['VISCODE'], row['RID'], row.get('PHASE', None)),
            axis=1
        )
        # Drop screen fails and unresolvable visits
        dfs_filtered[name] = dfs_filtered[name][dfs_filtered[name]['MONTH'] >= 0]

In [ ]:
df_master = (
    dfs_filtered['DXSUM'][["RID", "MONTH", "DIAGNOSIS", "DXAD", "DXMCI", "EXAMDATE"]]
    .sort_values(['RID', 'MONTH', 'DIAGNOSIS'])   # or sort by PHASE priority if you prefer
    .drop_duplicates(subset=['RID', 'MONTH'], keep='first')
    .copy()
)
demog = dfs_filtered['PTDEMOG'].drop_duplicates(subset=['RID'], keep='first')
df_master = df_master.merge(demog, on='RID', how='left')
df_master['AGE'] = df_master.apply(lambda row: (pd.to_datetime(row['EXAMDATE']) - pd.to_datetime(row['PTDOB'])).days / 365.25, axis=1)
df_master = df_master.drop(columns=['EXAMDATE', 'PTDOB'])
apoe = dfs_filtered['APOERES'][['RID', 'GENOTYPE']].drop_duplicates(subset=['RID'], keep='first')
df_master = df_master.merge(apoe, on='RID', how='left')
adas = dfs_filtered['ADAS'][["RID","MONTH","TOTSCORE","TOTAL13"]].drop_duplicates(subset=['RID', 'MONTH'], keep='first')
df_master = df_master.merge(adas, on=['RID', 'MONTH'], how='left')
faq = dfs_filtered['FAQ'][["RID","FAQTOTAL"  ]].drop_duplicates(subset=['RID'], keep='first')
df_master = df_master.merge(faq, on='RID', how='left')
fox = dfs_filtered['FOXLABBSI'][["RID","MONTH","BRAINVOL","VENTVOL"]].drop_duplicates(subset=['RID', 'MONTH'], keep='first')
df_master = df_master.merge(fox, on=['RID', 'MONTH'], how='left')
mmse = dfs_filtered['MMSE'][["RID","MONTH","MMSCORE"]].drop_duplicates(subset=['RID', 'MONTH'], keep='first')
df_master = df_master.merge(mmse, on=['RID', 'MONTH'], how='left')
cdr = dfs_filtered['CDR'][["RID","MONTH","CDRSB"]].drop_duplicates(subset=['RID', 'MONTH'], keep='first')
df_master = df_master.merge(cdr, on=['RID', 'MONTH'], how='left')

In [ ]:
VISCODE_PRIORITY = {
    'bl': 0, 'v03': 1, 'init': 2,
    '4_bl': 3, '4_init': 4,
    'sc': 5, 'v01': 6, '4_sc': 7,
    'scmri': 8, 'v02': 9,
}

neuro_cols = [
    "RID", "MONTH", "VISCODE",
    "LIMMTOTAL", "LDELTOTAL",
    "AVTOT1", "AVTOT2", "AVTOT3", "AVTOT4", "AVTOT5", "AVTOT6",
    "AVDEL30MIN", "AVDELTOT",
    "CATANIMSC",
]

neuro = dfs_filtered['NEUROBAT'].copy()
neuro['VISCODE_PRIORITY'] = (
    neuro['VISCODE'].str.strip().str.lower()
    .map(VISCODE_PRIORITY)
)

available_cols = [c for c in neuro_cols if c in neuro.columns] + ['VISCODE_PRIORITY']
neuro = neuro[available_cols]

neuro = (
    neuro
    .sort_values(['RID', 'MONTH', 'VISCODE_PRIORITY'])
    .groupby(['RID', 'MONTH'], as_index=False)
    .first()
    .drop(columns=['VISCODE', 'VISCODE_PRIORITY'])
)

df_master = df_master.merge(neuro, on=['RID', 'MONTH'], how='left')

ucd_wmh = dfs_filtered['UCD_WMH'].copy()
ucd_wmh['VISCODE_PRIORITY'] = (
    ucd_wmh['VISCODE'].str.strip().str.lower()
    .map(VISCODE_PRIORITY)
)
ucd_wmh = (
    ucd_wmh[["RID", "MONTH", "VISCODE", "TOTAL_HIPPO", "VISCODE_PRIORITY"]]
    .sort_values(['RID', 'MONTH', 'VISCODE_PRIORITY'])
    .groupby(['RID', 'MONTH'], as_index=False)
    .first()
    .drop(columns=['VISCODE', 'VISCODE_PRIORITY'])
)
df_master = df_master.merge(ucd_wmh, on=['RID', 'MONTH'], how='left')

In [ ]:
def apoe4_count(genotype):
    if pd.isna(genotype):
        return None
    geno_str = str(genotype).strip()
    import re
    if re.match(r'^\d/\d$', geno_str):
        return geno_str.split('/').count('4')
    else:
        return None

df_master['APOE4_COUNT'] = df_master['GENOTYPE'].apply(apoe4_count)
df_master.drop(columns=['GENOTYPE'], inplace=True)

In [ ]:
def label_progression(df):
    # Get worst diagnosis per RID across all months
    worst = df.groupby('RID')['DIAGNOSIS'].max().reset_index().rename(columns={'DIAGNOSIS': 'WORST_DX'})

    # Create output dataframe
    prog = worst[['RID']].copy()
    prog['PROGRESSOR'] = (worst['WORST_DX'] == 3).astype(int)

    return prog[['RID', 'PROGRESSOR']]

future_dx = label_progression(df_master)

In [ ]:
baseline_df = df_master[df_master['MONTH'] == 0].copy()

In [ ]:
baseline_df = (
    baseline_df
    .drop_duplicates(subset='RID', keep='first')
)

In [ ]:
baseline_df = baseline_df.merge(future_dx, on='RID', how='left')
baseline_df.drop(columns=['DXAD', 'DXMCI'], inplace=True)

In [ ]:
final_dataset = baseline_df.copy()
final_dataset_NOAD = final_dataset[final_dataset['DIAGNOSIS'] != 3].copy()

In [ ]:
for col in baseline_df.columns:
        n = baseline_df[col].notna().sum()
        pct = n / len(baseline_df) * 100
        print(f"  {col:<25} {n:>5} / {len(baseline_df)}  ({pct:.1f}%)")

print(f"\n=== CLASS BALANCE ===")
print(baseline_df['PROGRESSOR'].value_counts())
print(f"Progression rate: {baseline_df['PROGRESSOR'].mean()*100:.1f}%")

print(f"\n=== DIAGNOSIS BREAKDOWN ===")
print(final_dataset_NOAD['DIAGNOSIS'].map({1:'CU', 2:'MCI',3:'AD'}).value_counts())

  RID                        2937 / 2937  (100.0%)
  MONTH                      2937 / 2937  (100.0%)
  DIAGNOSIS                  2925 / 2937  (99.6%)
  PTGENDER                   2935 / 2937  (99.9%)
  PTEDUCAT                   2932 / 2937  (99.8%)
  PTMARRY                    2933 / 2937  (99.9%)
  AGE                        2903 / 2937  (98.8%)
  TOTSCORE                   2437 / 2937  (83.0%)
  TOTAL13                    2427 / 2937  (82.6%)
  FAQTOTAL                   2433 / 2937  (82.8%)
  BRAINVOL                   2511 / 2937  (85.5%)
  VENTVOL                    1619 / 2937  (55.1%)
  MMSCORE                    2914 / 2937  (99.2%)
  CDRSB                      2902 / 2937  (98.8%)
  LIMMTOTAL                  2918 / 2937  (99.4%)
  LDELTOTAL                  2915 / 2937  (99.3%)
  AVTOT1                     2450 / 2937  (83.4%)
  AVTOT2                     2450 / 2937  (83.4%)
  AVTOT3                     2450 / 2937  (83.4%)
  AVTOT4                     2449 / 2937  (83.4%

In [ ]:
for col in final_dataset_NOAD.columns:
        n = final_dataset_NOAD[col].notna().sum()
        pct = n / len(final_dataset_NOAD) * 100
        print(f"  {col:<25} {n:>5} / {len(final_dataset_NOAD)}  ({pct:.1f}%)")

print(f"\n=== CLASS BALANCE ===")
print(final_dataset_NOAD['PROGRESSOR'].value_counts())
print(f"Progression rate: {final_dataset_NOAD['PROGRESSOR'].mean()*100:.1f}%")

print(f"\n=== DIAGNOSIS BREAKDOWN ===")
print(final_dataset_NOAD['DIAGNOSIS'].map({1:'CU', 2:'MCI',3:'AD'}).value_counts())

  RID                        2535 / 2535  (100.0%)
  MONTH                      2535 / 2535  (100.0%)
  DIAGNOSIS                  2523 / 2535  (99.5%)
  PTGENDER                   2534 / 2535  (100.0%)
  PTEDUCAT                   2531 / 2535  (99.8%)
  PTMARRY                    2532 / 2535  (99.9%)
  AGE                        2504 / 2535  (98.8%)
  TOTSCORE                   2096 / 2535  (82.7%)
  TOTAL13                    2090 / 2535  (82.4%)
  FAQTOTAL                   2088 / 2535  (82.4%)
  BRAINVOL                   2171 / 2535  (85.6%)
  VENTVOL                    1333 / 2535  (52.6%)
  MMSCORE                    2517 / 2535  (99.3%)
  CDRSB                      2505 / 2535  (98.8%)
  LIMMTOTAL                  2520 / 2535  (99.4%)
  LDELTOTAL                  2520 / 2535  (99.4%)
  AVTOT1                     2107 / 2535  (83.1%)
  AVTOT2                     2107 / 2535  (83.1%)
  AVTOT3                     2107 / 2535  (83.1%)
  AVTOT4                     2106 / 2535  (83.1

In [ ]:
final_dataset.to_csv(processed_path + 'Dataset_AD.csv')
final_dataset_NOAD.to_csv(processed_path + 'Dataset.csv')

In [ ]:
print(pd.crosstab(final_dataset['DIAGNOSIS'], final_dataset['PROGRESSOR']))

PROGRESSOR     0    1
DIAGNOSIS            
1.0         1383   12
2.0          909  219
3.0            0  402


In [ ]:
print(pd.crosstab(final_dataset_NOAD['DIAGNOSIS'], final_dataset_NOAD['PROGRESSOR']))

PROGRESSOR     0    1
DIAGNOSIS            
1.0         1383   12
2.0          909  219
